# ⚙️ Notebook 03: Model Merging & GGUF Quantization for Edge Deployment
### Menggabungkan Adapter LoRA dan Mengonversi ke Format GGUF untuk Raspberry Pi 5 / PC / Android

Notebook ini menggabungkan bobot LoRA adapter ke base model, lalu mengompilasi model ke format **GGUF (Q4_K_M)** menggunakan `llama.cpp`.

In [ ]:
# 1. Hubungkan Google Drive tempat penyimpanan adapter
from google.colab import drive
drive.mount('/content/drive')

import os
WORK_DIR = '/content/drive/MyDrive/Edge-VLM-Project'
ADAPTER_DIR = f"{WORK_DIR}/models/edge_vlm_final_adapter"
MERGED_DIR = f"{WORK_DIR}/models/edge_vlm_merged_fp16"
GGUF_DIR = f"{WORK_DIR}/models/gguf_export"
os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

In [ ]:
# 2. Install dependensi dan clone llama.cpp
!pip install -q peft transformers torch accelerate
!git clone https://github.com/ggerganov/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && make -j$(nproc)

In [ ]:
# 3. Gabungkan LoRA Adapter ke Base Model (Full Precision Merge)
import torch
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor
from peft import PeftModel

BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
print("⏳ Memuat base model FP16...")
base_model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="cpu"
)

print("⏳ Menggabungkan adapter...")
merged_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged_model = merged_model.merge_and_unload()

print(f"💾 Menyimpan model gabungan ke: {MERGED_DIR}...")
merged_model.save_pretrained(MERGED_DIR)
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
processor.save_pretrained(MERGED_DIR)
print("✅ Model merged FP16 berhasil disimpan!")

In [ ]:
# 4. Konversi ke Format GGUF menggunakan llama.cpp converter
!python3 /content/llama.cpp/convert_hf_to_gguf.py {MERGED_DIR} --outfile {GGUF_DIR}/edge-vlm-diag-f16.gguf --outtype f16

# 5. Kuantisasi ke Q4_K_M (Ukuran hanya ~1.4 GB, super ringan untuk Raspberry Pi 5)
!/content/llama.cpp/llama-quantize {GGUF_DIR}/edge-vlm-diag-f16.gguf {GGUF_DIR}/edge-vlm-diag-q4_k_m.gguf q4_k_m

print("\n" + "="*60)
print(f"🎉 FILE MODEL GGUF SIAP DIUNDUH: {GGUF_DIR}/edge-vlm-diag-q4_k_m.gguf")
print("Model ini bisa langsung dijalankan di Raspberry Pi 5 atau Laptop via Ollama / llama.cpp!")
print("="*60)